In [5]:
# === CELL 1: Imports & Setup ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import os
import requests

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === CELL 2: Data Loading (Penn TreeBank Word Level) ===
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []

    def add_word(self, word):
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
        return self.word2idx[word]

    def __len__(self):
        return len(self.idx2word)

class Corpus:
    def __init__(self, path):
        self.dictionary = Dictionary()
        self.train = self.tokenize(os.path.join(path, 'ptb.train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'ptb.valid.txt'))
        self.test = self.tokenize(os.path.join(path, 'ptb.test.txt'))

    def tokenize(self, path):
        assert os.path.exists(path)
        with open(path, 'r', encoding="utf8") as f:
            for line in f:
                words = line.split() + ['<eos>']
                for word in words:
                    self.dictionary.add_word(word)
        with open(path, 'r', encoding="utf8") as f:
            idss = []
            for line in f:
                words = line.split() + ['<eos>']
                ids = []
                for word in words:
                    ids.append(self.dictionary.word2idx[word])
                idss.append(torch.tensor(ids).type(torch.int64))
            ids = torch.cat(idss)
        return ids

def batchify(data, bsz):
    nbatch = data.size(0) // bsz
    data = data.narrow(0, 0, nbatch * bsz)
    data = data.view(bsz, -1).t().contiguous()
    return data.to(device)

def get_batch(source, i, seq_len):
    seq_len = min(seq_len, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].view(-1)
    return data, target

def download_ptb():
    base_url = "https://raw.githubusercontent.com/tomsercu/lstm/master/data"
    files = ["ptb.train.txt", "ptb.valid.txt", "ptb.test.txt"]
    os.makedirs("data/ptb", exist_ok=True)
    for filename in files:
        path = f"data/ptb/{filename}"
        if not os.path.exists(path) or os.path.getsize(path) < 100:
            print(f"Downloading {filename}...")
            r = requests.get(f"{base_url}/{filename}")
            with open(path, 'wb') as f:
                f.write(r.content)

download_ptb()
corpus = Corpus('data/ptb')
vocab_size = len(corpus.dictionary)
print(f"Vocab size: {vocab_size}")

# === CELL 3: INNv2 Architecture (Regularized) ===

class MultiMambaBlock(nn.Module):
    def __init__(self, num_neurons, d_model, d_state=16, d_conv=4, expand=2, dropout=0.1):
        super().__init__()
        self.num_neurons = num_neurons
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.d_state = d_state
        self.dt_rank = math.ceil(d_model / 16)

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        self.conv1d = nn.Conv1d(
            in_channels=num_neurons * self.d_inner,
            out_channels=num_neurons * self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=num_neurons * self.d_inner,
            padding=d_conv - 1,
        )

        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.dropout = nn.Dropout(dropout) # Added Dropout

    def forward(self, x):
        B, N, L, D = x.shape
        x_and_res = self.in_proj(x)
        (x_in, res) = x_and_res.split(split_size=[self.d_inner, self.d_inner], dim=-1)

        x_conv = x_in.permute(0, 1, 3, 2).reshape(B, N*self.d_inner, L)
        x_conv = self.conv1d(x_conv)[:, :, :L]
        x_conv = x_conv.reshape(B, N, self.d_inner, L).permute(0, 1, 3, 2)
        x_conv = F.silu(x_conv)

        y = self.ssm(x_conv)
        y = y * F.silu(res)
        out = self.out_proj(y)
        return self.dropout(out) # Apply Dropout

    def ssm(self, x):
        # Robust FP32 SSM
        x_flat = x.reshape(-1, x.size(2), x.size(3))
        dt_rank_state = self.x_proj(x_flat)
        dt, B, C = torch.split(dt_rank_state, [self.dt_rank, self.d_state, self.d_state], dim=-1)

        dt = self.dt_proj(dt)
        dt = F.softplus(dt)

        A = -torch.exp(self.A_log.float())

        y = []
        h = torch.zeros(x_flat.size(0), self.d_inner, self.d_state, device=x.device)

        dA = torch.exp(torch.einsum('bld,ds->blds', dt, A))
        dB = torch.einsum('bld,bls->blds', dt, B)

        for t in range(x.size(2)):
            u = x_flat[:, t, :]
            h = dA[:, t, :, :] * h + dB[:, t, :, :] * u.unsqueeze(-1)
            y_t = torch.einsum('bds,bs->bd', h, C[:, t, :])
            y.append(y_t)

        y = torch.stack(y, dim=1)
        y = y + x_flat * self.D
        return y.reshape(x.shape)

class MultiHeadNeuronAttention(nn.Module):
    def __init__(self, num_neurons, d_model, n_head, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_head, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        res = x
        x, _ = self.attn(x, x, x)
        return self.norm(res + self.dropout(x))

class ParallelINN(nn.Module):
    def __init__(self, vocab_size, num_neurons, d_model, num_layers, n_head=4, dropout=0.1):
        super().__init__()
        self.num_neurons = num_neurons
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(dropout) # Embedding dropout

        self.layers = nn.ModuleList([])
        for _ in range(num_layers):
            mamba = MultiMambaBlock(num_neurons, d_model, dropout=dropout)
            attn = MultiHeadNeuronAttention(num_neurons, d_model, n_head, dropout=dropout)
            self.layers.append(nn.ModuleList([mamba, attn]))

        self.norm_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids):
        B, L = input_ids.shape
        x = self.embedding(input_ids)
        x = self.dropout(x) # Apply dropout to embeddings

        x = x.unsqueeze(1).expand(-1, self.num_neurons, -1, -1).contiguous()

        for mamba, attn in self.layers:
            x = x + mamba(x)
            x_flat = x.permute(0, 2, 1, 3).reshape(B*L, self.num_neurons, -1)
            x_flat = attn(x_flat)
            x = x_flat.view(B, L, self.num_neurons, -1).permute(0, 2, 1, 3)

        out = x.mean(dim=1)
        out = self.norm_f(out)
        return self.head(out)

# === CELL 4: Configuration & Training (Regularized) ===

CONFIG = {
    'd_model': 256,
    'num_neurons': 16,
    'num_layers': 4,
    'lr': 3e-4,
    'batch_size': 16,
    'seq_len': 64,
    'epochs': 20,
    'grad_clip': 1.0,
    'dropout': 0.3,      # HIGH DROPOUT FOR PTB
    'weight_decay': 0.1  # STRONG REGULARIZATION
}

print(f"=== STARTING TRAINING: INNv2 REGULARIZED MODE ===")
print(f"Config: {CONFIG}")

model = ParallelINN(
    vocab_size,
    CONFIG['num_neurons'],
    CONFIG['d_model'],
    CONFIG['num_layers'],
    dropout=CONFIG['dropout']
).to(device)

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CONFIG['lr'],
    steps_per_epoch=len(corpus.train) // (CONFIG['batch_size']*CONFIG['seq_len']),
    epochs=CONFIG['epochs']
)
criterion = nn.CrossEntropyLoss()

train_data = batchify(corpus.train, CONFIG['batch_size'])
val_data = batchify(corpus.valid, CONFIG['batch_size'])

def evaluate(source):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for i in range(0, source.size(0) - 1, CONFIG['seq_len']):
            data, targets = get_batch(source, i, CONFIG['seq_len'])
            output = model(data)
            total_loss += len(data) * criterion(output.reshape(-1, vocab_size), targets).item()
    return total_loss / (len(source) - 1)

try:
    for epoch in range(1, CONFIG['epochs'] + 1):
        model.train()
        total_loss = 0
        start_time = time.time()
        num_batches = train_data.size(0) // CONFIG['seq_len']

        for batch, i in enumerate(range(0, train_data.size(0) - 1, CONFIG['seq_len'])):
            data, targets = get_batch(train_data, i, CONFIG['seq_len'])

            output = model(data)
            loss = criterion(output.reshape(-1, vocab_size), targets)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

            if batch % 50 == 0 and batch > 0:
                cur_loss = total_loss / 50
                print(f'| epoch {epoch} | {batch}/{num_batches} | lr {scheduler.get_last_lr()[0]:.5f} | '
                      f'loss {cur_loss:.2f} | ppl {math.exp(cur_loss):.2f}')
                total_loss = 0

        val_loss = evaluate(val_data)
        print(f'-' * 89)
        print(f'| end of epoch {epoch} | valid loss {val_loss:.2f} | valid ppl {math.exp(val_loss):.2f}')
        print(f'-' * 89)

except KeyboardInterrupt:
    print("Training interrupted.")


Using device: cuda
Vocab size: 10000
=== STARTING TRAINING: INNv2 REGULARIZED MODE ===
Config: {'d_model': 256, 'num_neurons': 16, 'num_layers': 4, 'lr': 0.0003, 'batch_size': 16, 'seq_len': 64, 'epochs': 20, 'grad_clip': 1.0, 'dropout': 0.3, 'weight_decay': 0.1}
Params: 5,519,872
| epoch 1 | 50/907 | lr 0.00001 | loss 9.38 | ppl 11830.30
| epoch 1 | 100/907 | lr 0.00001 | loss 9.12 | ppl 9105.43
| epoch 1 | 150/907 | lr 0.00001 | loss 9.05 | ppl 8479.62
| epoch 1 | 200/907 | lr 0.00001 | loss 8.91 | ppl 7377.51
| epoch 1 | 250/907 | lr 0.00001 | loss 8.70 | ppl 6007.86
| epoch 1 | 300/907 | lr 0.00001 | loss 8.46 | ppl 4743.30
| epoch 1 | 350/907 | lr 0.00001 | loss 8.19 | ppl 3592.99
| epoch 1 | 400/907 | lr 0.00002 | loss 7.92 | ppl 2748.18
| epoch 1 | 450/907 | lr 0.00002 | loss 7.69 | ppl 2188.48
| epoch 1 | 500/907 | lr 0.00002 | loss 7.46 | ppl 1743.10
| epoch 1 | 550/907 | lr 0.00002 | loss 7.28 | ppl 1448.94
| epoch 1 | 600/907 | lr 0.00002 | loss 7.10 | ppl 1216.37
| epoch 1 

ValueError: Tried to step 18141 times. The specified number of total steps is 18140

In [6]:
# === CELLULE FINALE : SAUVEGARDE & GÉNÉRATION ===

# 1. Sauvegarde du Modèle
torch.save(model.state_dict(), "innv2_ptb_word_level.pth")
print("Model saved successfully as 'innv2_ptb_word_level.pth'")

# 2. Fonction de Génération
def generate_text(model, seed_text, length=50, temperature=0.8):
    model.eval()
    words = seed_text.lower().split()
    ids = [corpus.dictionary.word2idx.get(w, corpus.dictionary.word2idx['<unk>']) for w in words]
    input_tensor = torch.tensor([ids], dtype=torch.long).to(device)

    generated = words[:]

    with torch.no_grad():
        for _ in range(length):
            output = model(input_tensor)
            # On prend le dernier token
            logits = output[0, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)

            # Sampling
            next_token = torch.multinomial(probs, 1).item()
            next_word = corpus.dictionary.idx2word[next_token]

            generated.append(next_word)
            input_tensor = torch.cat([input_tensor, torch.tensor([[next_token]], device=device)], dim=1)

    return " ".join(generated)

# 3. Test de génération
print("\n=== GENERATED TEXT SAMPLES ===")
seeds = ["the market", "investors are", "despite the loss"]
for seed in seeds:
    print(f"\nPrompt: '{seed}'")
    print(f"Generated: {generate_text(model, seed)}")

Model saved successfully as 'innv2_ptb_word_level.pth'

=== GENERATED TEXT SAMPLES ===

Prompt: 'the market'
Generated: the market an organized <unk> over a share up $ N N billion and N <eos> it introduced by N N N <eos> it received by <unk> of the report which citizens who would have a member here <eos> the difference compares with the <unk> sales would be <unk> in recent weeks

Prompt: 'investors are'
Generated: investors are running at one facilities <eos> revenue which it is a <unk> and the past two years ago <eos> independent producers ' support of return <eos> thousands of a better than N N million or less probable market <unk> <eos> but i do n't disclosed <eos> the N people were priced

Prompt: 'despite the loss'
Generated: despite the loss of the foreign military program was awarded its rolling senators <eos> but they 're check to N N N N billion in the american express the fourth quarter included big board of <unk> field near the only for the current weakness in the <unk> <eos

## Comparaison aux SOTA

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import os
import requests

# === CONFIGURATION ===
CONFIG = {
    'vocab_size': 10000,
    'd_model': 256,
    'n_layers': 4,
    'n_head': 4,
    'd_hid': 1024,     # Standard FFN ratio
    'lstm_layers': 2,
    'dropout': 0.1,
    'lr': 3e-4,       # Standard Adam LR
    'batch_size': 16,
    'seq_len': 64,
    'epochs': 20
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === DATA LOADING ===
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
    def add_word(self, word):
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
        return self.word2idx[word]
    def __len__(self): return len(self.idx2word)

class Corpus:
    def __init__(self, path):
        self.dictionary = Dictionary()
        self.train = self.tokenize(os.path.join(path, 'ptb.train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'ptb.valid.txt'))
    def tokenize(self, path):
        with open(path, 'r', encoding="utf8") as f:
            for line in f:
                words = line.split() + ['<eos>']
                for word in words: self.dictionary.add_word(word)
        with open(path, 'r', encoding="utf8") as f:
            idss = []
            for line in f:
                words = line.split() + ['<eos>']
                ids = [self.dictionary.word2idx[w] for w in words]
                idss.append(torch.tensor(ids).type(torch.int64))
            ids = torch.cat(idss)
        return ids

def batchify(data, bsz):
    nbatch = data.size(0) // bsz
    data = data.narrow(0, 0, nbatch * bsz)
    data = data.view(bsz, -1).t().contiguous()
    return data.to(device)

def get_batch(source, i, seq_len):
    seq_len = min(seq_len, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].view(-1)
    return data, target

def download_ptb():
    base_url = "https://raw.githubusercontent.com/tomsercu/lstm/master/data"
    files = ["ptb.train.txt", "ptb.valid.txt"]
    os.makedirs("data/ptb", exist_ok=True)
    for filename in files:
        path = f"data/ptb/{filename}"
        if not os.path.exists(path):
            r = requests.get(f"{base_url}/{filename}")
            with open(path, 'wb') as f: f.write(r.content)

# === BASELINE MODELS ===

class LSTMBaseline(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.lstm = nn.LSTM(d_model, d_model, n_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.fc.weight = self.embedding.weight # Tying
        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()
        self.fc.weight.data.uniform_(-initrange, initrange)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        return self.fc(out)

# STANDARD TRANSFORMER (FIXED INIT)
class TransformerBaseline(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, d_hid, n_layers, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # Standard PyTorch Transformer Layer
        encoder_layer = nn.TransformerEncoderLayer(d_model, n_head, d_hid, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, n_layers)
        self.fc = nn.Linear(d_model, vocab_size)

        self.fc.weight = self.embedding.weight # Tying
        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()
        self.fc.weight.data.uniform_(-initrange, initrange)

    def forward(self, src):
        # Mask for causality (crucial for LM)
        mask = nn.Transformer.generate_square_subsequent_mask(src.size(1)).to(src.device)

        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src, mask, is_causal=True)
        return self.fc(output)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# === TRAINING LOOP ===
def run_benchmark(model_class, name, **kwargs):
    print(f"\n>>> STARTING BENCHMARK: {name}")
    model = model_class(CONFIG['vocab_size'], CONFIG['d_model'], **kwargs).to(device)
    params = sum(p.numel() for p in model.parameters())
    print(f"Model Params: {params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CONFIG['lr'],
        steps_per_epoch=len(corpus.train) // (CONFIG['batch_size']*CONFIG['seq_len']),
        epochs=CONFIG['epochs']
    )
    criterion = nn.CrossEntropyLoss()

    train_data = batchify(corpus.train, CONFIG['batch_size'])
    val_data = batchify(corpus.valid, CONFIG['batch_size'])

    for epoch in range(1, CONFIG['epochs'] + 1):
        model.train()
        total_loss = 0
        start_time = time.time()

        for batch, i in enumerate(range(0, train_data.size(0) - 1, CONFIG['seq_len'])):
            data, targets = get_batch(train_data, i, CONFIG['seq_len'])
            output = model(data)
            loss = criterion(output.reshape(-1, CONFIG['vocab_size']), targets)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for i in range(0, val_data.size(0) - 1, CONFIG['seq_len']):
                data, targets = get_batch(val_data, i, CONFIG['seq_len'])
                output = model(data)
                val_loss += len(data) * criterion(output.reshape(-1, CONFIG['vocab_size']), targets).item()
        val_loss /= (len(val_data) - 1)

        train_ppl = math.exp(total_loss / (len(train_data) // CONFIG['seq_len']))
        valid_ppl = math.exp(val_loss)

        print(f"| Epoch {epoch:2d} | Train PPL: {train_ppl:6.2f} | Valid PPL: {valid_ppl:6.2f} | Time: {time.time() - start_time:.0f}s")

if __name__ == "__main__":
    download_ptb()
    corpus = Corpus('data/ptb')

    # 1. LSTM (6 layers -> ~5.6M Params)
    # run_benchmark(LSTMBaseline, "LSTM Baseline", n_layers=6)

    # 2. Transformer (4 layers, 1024 FFN -> ~5.8M Params)
    run_benchmark(TransformerBaseline, "Transformer Baseline (FIXED)", n_layers=4, n_head=4, d_hid=1024)



>>> STARTING BENCHMARK: Transformer Baseline (FIXED)
Model Params: 5,729,040
| Epoch  1 | Train PPL: 1424.71 | Valid PPL: 631.98 | Time: 10s
| Epoch  2 | Train PPL: 529.51 | Valid PPL: 409.16 | Time: 10s
| Epoch  3 | Train PPL: 357.83 | Valid PPL: 294.42 | Time: 10s
| Epoch  4 | Train PPL: 266.03 | Valid PPL: 247.09 | Time: 10s
| Epoch  5 | Train PPL: 218.42 | Valid PPL: 225.64 | Time: 10s
| Epoch  6 | Train PPL: 189.78 | Valid PPL: 215.78 | Time: 10s
| Epoch  7 | Train PPL: 169.91 | Valid PPL: 211.87 | Time: 10s
| Epoch  8 | Train PPL: 155.32 | Valid PPL: 210.45 | Time: 10s
| Epoch  9 | Train PPL: 144.00 | Valid PPL: 211.19 | Time: 10s
| Epoch 10 | Train PPL: 134.55 | Valid PPL: 212.31 | Time: 10s
| Epoch 11 | Train PPL: 126.61 | Valid PPL: 214.43 | Time: 10s
| Epoch 12 | Train PPL: 119.79 | Valid PPL: 216.83 | Time: 10s
| Epoch 13 | Train PPL: 114.35 | Valid PPL: 219.02 | Time: 10s
| Epoch 14 | Train PPL: 110.11 | Valid PPL: 220.11 | Time: 10s
| Epoch 15 | Train PPL: 106.38 | Valid 

ValueError: Tried to step 18141 times. The specified number of total steps is 18140